In [ ]:
"""
Partial Dependence Plots (1D) and 2D Interaction Plots
=======================================================
Top-ranked variables: WaterTemp, pH, OrgN
Variable pairs:       WaterTemp×pH, WaterTemp×OrgN, pH×OrgN
Models:               Logistic Regression, Decision Tree, XGBoost,
                      Random Forest, ANN

Procedure
---------
1. Load the same CSV used in the CV/LOSO analyses.
2. Refit each model on the FULL scaled dataset (no train/test split).
3. Compute marginal PDPs using sklearn's PartialDependenceDisplay /
   partial_dependence for sklearn models; manual grid averaging for ANN.
4. Compute 2D PDPs (interaction surfaces) for all three variable pairs.
5. Save every figure as PNG + EPS under OUTPUT_DIR/figures/.

Notes
-----
- PDP values represent the marginal effect of one (or two) variables
  on the predicted probability of the "Risky" class, averaged over
  the empirical distribution of all other features.
- For tree-based and linear models sklearn's partial_dependence() is
  used directly.  For the ANN a manual ICE/PDP grid is computed.
- Colormaps follow the same model colour palette as the CV pipeline.
"""

# =============================================================================
# IMPORTS
# =============================================================================
import os
# Run from the notebooks/ directory OR set kernel working directory to HAB_Phase1/
import sys, pathlib


import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
plt.ioff()

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from sklearn.inspection      import partial_dependence
from sklearn.preprocessing   import StandardScaler

import tensorflow as tf
import tensorflow.keras as keras

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION  — edit paths / columns here
# =============================================================================

PROJECT_ROOT = pathlib.Path().resolve()
while PROJECT_ROOT.name != "HAB_Phase1":
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find HAB_Phase1 project root")

sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, OUTPUT_DIRS

# DATA_PATH    = "Data/UCDavis_Data_for_Analysis2_220days_ToxinProdMicrocystis.csv"
# OUTPUT_DIR   = "C:/Temp/PDP_Results"
OUTPUT_DIR = str(OUTPUT_DIRS["05"])  # change for each notebook
DATA_PATH  = str(DATA_PATH)

TARGET_COL   = "ToxinProdc_Microcystis_CC"
THRESHOLD    = 4000
RANDOM_STATE = 42

FEATURES = ["Q", "Q_1m", "SpecCond.", "OrgC", "OrgN", "PO4",
            "WaterTemp", "DO", "pH"]

# Top-ranked variables for PDP / interaction analysis
TOP_VARS   = ["WaterTemp", "pH", "OrgN"]
TOP_PAIRS  = [("WaterTemp", "pH"), ("WaterTemp", "OrgN"), ("pH", "OrgN")]

# Grid resolution for 2D surfaces
GRID_RESOLUTION = 50

# ANN hyperparameters (must match the training pipeline)
ANN_HIDDEN_LAYERS = 3
ANN_ACTIVATION    = "elu"
ANN_NEURONS       = 128
ANN_LEARNING_RATE = 0.0081
ANN_DROPOUT_RATE  = 0.25
ANN_EPOCHS        = 1000
ANN_BATCH_SIZE    = 32
ANN_VAL_SPLIT     = 0.2
ANN_PATIENCE      = 30

ALL_MODELS = ["Logistic Regression", "Decision Tree", "XGBoost",
              "Random Forest", "ANN"]

COLORS = {
    "Logistic Regression": "#9467BD",
    "Decision Tree":       "#8C564B",
    "XGBoost":             "#B85042",
    "Random Forest":       "#1C7293",
    "ANN":                 "#E8A838",
}

# =============================================================================
# HELPERS
# =============================================================================
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def setup_dirs():
    for sub in ("figures/pdp_1d", "figures/pdp_2d"):
        os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)
    print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")


def savefig(fig, subfolder: str, filename: str):
    folder = os.path.abspath(os.path.join(OUTPUT_DIR, "figures", subfolder))
    os.makedirs(folder, exist_ok=True)
    for ext in ("png", "eps"):
        path = os.path.join(folder, f"{filename}.{ext}")
        try:
            fig.savefig(path, dpi=150, bbox_inches="tight",
                        format=ext if ext == "eps" else None)
            print(f"  Saved → {path}")
        except Exception as e:
            print(f"  [WARN] Could not save {path}: {e}")


# =============================================================================
# DATA LOADING
# =============================================================================
def load_data():
    df = pd.read_csv(DATA_PATH)
    df["label"] = (df[TARGET_COL] >= THRESHOLD).astype(int)
    df = df.dropna(subset=FEATURES + ["label"])
    X = df[FEATURES].copy()
    y = df["label"].values
    print(f"Loaded {len(y)} samples | Positive (Risky): {y.sum()} "
          f"({100*y.mean():.1f}%)")
    return X, y


# =============================================================================
# MODEL BUILDERS
# =============================================================================
def build_lr():
    return LogisticRegression(
        warm_start=False, tol=0.0001, solver="saga", penalty="l1",
        max_iter=1000, fit_intercept=True, class_weight="balanced",
        C=1, random_state=RANDOM_STATE,
    )

def build_dt():
    return DecisionTreeClassifier(
        splitter="random", min_samples_split=5, min_samples_leaf=2,
        min_impurity_decrease=0.0, max_leaf_nodes=None, max_features=0.5,
        max_depth=5, criterion="entropy", class_weight="balanced",
        ccp_alpha=0.01, random_state=RANDOM_STATE,
    )

def build_xgb():
    return XGBClassifier(
        subsample=0.8, scale_pos_weight=1.0, n_estimators=200,
        min_child_weight=3, max_depth=2, learning_rate=0.05,
        reg_lambda=1.0, gamma=0.3, colsample_bytree=0.5,
        alpha=0.1, colsample_bylevel=0.8278, colsample_bynode=0.7501,
        max_delta_step=0, random_state=RANDOM_STATE,
        verbosity=0, n_jobs=1, objective="binary:logistic",
    )

def build_rf():
    return RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_split=5,
        min_samples_leaf=8, max_features=0.3, bootstrap=True,
        class_weight="balanced", criterion="gini", ccp_alpha=0.0001,
        min_impurity_decrease=0.01627, min_weight_fraction_leaf=0.01946,
        max_leaf_nodes=57, random_state=RANDOM_STATE,
    )


# =============================================================================
# ANN WRAPPER
# =============================================================================
class ANNWrapper:
    def __init__(self, input_dim: int, seed: int = RANDOM_STATE):
        self.input_dim = input_dim
        self.seed      = seed
        self.model_    = None

    def _build(self):
        tf.random.set_seed(self.seed)
        m = keras.Sequential(name="ANN_HAB")
        m.add(keras.layers.Input(shape=(self.input_dim,)))
        m.add(keras.layers.Dense(ANN_NEURONS, activation=ANN_ACTIVATION))
        m.add(keras.layers.Dropout(ANN_DROPOUT_RATE, seed=self.seed))
        for _ in range(ANN_HIDDEN_LAYERS - 1):
            m.add(keras.layers.Dense(ANN_NEURONS, activation=ANN_ACTIVATION))
            m.add(keras.layers.Dropout(ANN_DROPOUT_RATE, seed=self.seed))
        m.add(keras.layers.Dense(1, activation="sigmoid"))
        m.compile(
            optimizer=keras.optimizers.Adam(learning_rate=ANN_LEARNING_RATE),
            loss="binary_crossentropy", metrics=["accuracy"],
        )
        return m

    def fit(self, X, y):
        self.model_ = self._build()
        cb = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=ANN_PATIENCE,
            restore_best_weights=True, verbose=0,
        )
        self.model_.fit(X, y, epochs=ANN_EPOCHS, batch_size=ANN_BATCH_SIZE,
                        validation_split=ANN_VAL_SPLIT,
                        callbacks=[cb], verbose=0)
        return self

    def predict_proba_positive(self, X):
        """Return P(Risky) for a 2-D array of scaled features."""
        return self.model_.predict(X, verbose=0).ravel()


# =============================================================================
# FIT ALL MODELS ON FULL DATA
# =============================================================================
def fit_all_models(X_sc: np.ndarray, y: np.ndarray):
    """
    Refit every model on the entire scaled dataset.
    Returns a dict of fitted model objects.
    """
    print("\nFitting models on full dataset ...")
    X_df = pd.DataFrame(X_sc, columns=FEATURES)

    fitted = {}

    for name, builder in [
        ("Logistic Regression", build_lr),
        ("Decision Tree",       build_dt),
        ("XGBoost",             build_xgb),
        ("Random Forest",       build_rf),
    ]:
        print(f"  Fitting {name} ...")
        m = builder()
        m.fit(X_df, y)
        fitted[name] = m

    print("  Fitting ANN ...")
    ann = ANNWrapper(X_sc.shape[1])
    ann.fit(X_sc, y)
    fitted["ANN"] = ann
    keras.backend.clear_session()
    gc.collect()

    print("  All models fitted.")
    return fitted


# =============================================================================
# PDP COMPUTATION
# =============================================================================
def pdp_1d_sklearn(model, X_df: pd.DataFrame, feature: str,
                   grid_resolution: int = GRID_RESOLUTION):
    """
    1-D PDP for a sklearn-compatible model.
    Returns (grid_values, avg_predictions).
    """
    feat_idx = list(X_df.columns).index(feature)
    result   = partial_dependence(
        model, X_df, features=[feat_idx],
        kind="average", grid_resolution=grid_resolution,
        response_method="predict_proba",
    )
    grid_vals = result["grid_values"][0]
    avg_preds = result["average"][0]     # shape (grid_resolution,)
    # partial_dependence returns both classes; take class-1 (Risky)
    if avg_preds.ndim > 1:
        avg_preds = avg_preds[1]
    return grid_vals, avg_preds


def pdp_1d_ann(ann: ANNWrapper, X_sc: np.ndarray, feature: str,
               grid_resolution: int = GRID_RESOLUTION):
    """
    Manual 1-D PDP for the ANN.
    """
    feat_idx  = FEATURES.index(feature)
    feat_vals = np.linspace(X_sc[:, feat_idx].min(),
                            X_sc[:, feat_idx].max(), grid_resolution)
    avg_preds = []
    X_copy = X_sc.copy()
    for val in feat_vals:
        X_copy[:, feat_idx] = val
        probs = ann.predict_proba_positive(X_copy)
        avg_preds.append(probs.mean())
    return feat_vals, np.array(avg_preds)


def pdp_2d_sklearn(model, X_df: pd.DataFrame, feat1: str, feat2: str,
                   grid_resolution: int = GRID_RESOLUTION):
    """
    2-D PDP for a sklearn-compatible model.
    Returns (grid1, grid2, avg_surface) where avg_surface is (G×G).
    """
    idx1 = list(X_df.columns).index(feat1)
    idx2 = list(X_df.columns).index(feat2)
    result = partial_dependence(
        model, X_df, features=[(idx1, idx2)],
        kind="average", grid_resolution=grid_resolution,
        response_method="predict_proba",
    )
    grid1   = result["grid_values"][0]
    grid2   = result["grid_values"][1]
    surface = result["average"][0]   # shape (2, G, G) or (G, G)
    if surface.ndim == 3:
        surface = surface[1]         # class-1 (Risky)
    return grid1, grid2, surface


def pdp_2d_ann(ann: ANNWrapper, X_sc: np.ndarray,
               feat1: str, feat2: str,
               grid_resolution: int = GRID_RESOLUTION):
    """
    Manual 2-D PDP for the ANN.
    """
    idx1 = FEATURES.index(feat1)
    idx2 = FEATURES.index(feat2)
    g1   = np.linspace(X_sc[:, idx1].min(), X_sc[:, idx1].max(), grid_resolution)
    g2   = np.linspace(X_sc[:, idx2].min(), X_sc[:, idx2].max(), grid_resolution)
    surface = np.empty((len(g1), len(g2)))
    X_copy  = X_sc.copy()
    for i, v1 in enumerate(g1):
        for j, v2 in enumerate(g2):
            X_copy[:, idx1] = v1
            X_copy[:, idx2] = v2
            surface[i, j]   = ann.predict_proba_positive(X_copy).mean()
    return g1, g2, surface


# =============================================================================
# BACK-TRANSFORM SCALED GRID TO ORIGINAL UNITS
# =============================================================================
def unscale_grid(grid_sc: np.ndarray, feature: str, scaler: StandardScaler):
    feat_idx = FEATURES.index(feature)
    mean     = scaler.mean_[feat_idx]
    std      = scaler.scale_[feat_idx]
    return grid_sc * std + mean


# =============================================================================
# FIGURE 1 — 1-D PARTIAL DEPENDENCE PLOTS
# One figure per top variable; subplots = one per model (5 subplots).
# =============================================================================
def plot_pdp_1d(fitted: dict, X_sc: np.ndarray, X_df: pd.DataFrame,
                scaler: StandardScaler):
    print("\n[Figure 1] 1-D Partial Dependence Plots ...")

    for feature in TOP_VARS:
        fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=False)
        fig.suptitle(
            f"1-D Partial Dependence — {feature}\n"
            f"(marginal effect on P(Risky), averaged over all other features)",
            fontsize=13, fontweight="bold", y=1.03,
        )

        for ax, mname in zip(axes, ALL_MODELS):
            color = COLORS[mname]
            if mname == "ANN":
                grid_sc, avg = pdp_1d_ann(fitted[mname], X_sc, feature)
            else:
                grid_sc, avg = pdp_1d_sklearn(fitted[mname], X_df, feature)

            # Back-transform x-axis to original units
            grid_orig = unscale_grid(grid_sc, feature, scaler)

            ax.plot(grid_orig, avg, color=color, linewidth=2.2)
            ax.fill_between(grid_orig, avg, alpha=0.15, color=color)
            ax.axhline(0.5, color="black", linestyle="--",
                       linewidth=1.0, label="Decision threshold")
            ax.set_title(mname, fontsize=10, fontweight="bold")
            ax.set_xlabel(feature, fontsize=9)
            ax.set_ylabel("P(Risky)" if ax == axes[0] else "", fontsize=9)
            ax.set_ylim(-0.05, 1.10)
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
            ax.yaxis.grid(True, linestyle="--", alpha=0.4)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)

        plt.tight_layout()
        savefig(fig, "pdp_1d", f"PDP1D_{feature.replace('.', '')}")
        plt.close(fig)
        gc.collect()

    # Combined figure: rows = variables, cols = models
    fig, axes = plt.subplots(len(TOP_VARS), len(ALL_MODELS),
                             figsize=(22, 4 * len(TOP_VARS)),
                             sharey="row")
    fig.suptitle(
        "1-D Partial Dependence Plots — All Top Variables × All Models\n"
        "(marginal effect on P(Risky), averaged over all other features)",
        fontsize=14, fontweight="bold", y=1.02,
    )
    for row, feature in enumerate(TOP_VARS):
        for col, mname in enumerate(ALL_MODELS):
            ax    = axes[row][col]
            color = COLORS[mname]
            if mname == "ANN":
                grid_sc, avg = pdp_1d_ann(fitted[mname], X_sc, feature)
            else:
                grid_sc, avg = pdp_1d_sklearn(fitted[mname], X_df, feature)
            grid_orig = unscale_grid(grid_sc, feature, scaler)
            ax.plot(grid_orig, avg, color=color, linewidth=2.0)
            ax.fill_between(grid_orig, avg, alpha=0.15, color=color)
            ax.axhline(0.5, color="black", linestyle="--", linewidth=0.9)
            ax.set_ylim(-0.05, 1.10)
            if row == 0:
                ax.set_title(mname, fontsize=10, fontweight="bold")
            if col == 0:
                ax.set_ylabel(f"{feature}\nP(Risky)", fontsize=9)
            ax.set_xlabel(feature if row == len(TOP_VARS) - 1 else "",
                          fontsize=8)
            ax.yaxis.grid(True, linestyle="--", alpha=0.4)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    savefig(fig, "pdp_1d", "PDP1D_combined")
    plt.close(fig)
    gc.collect()
    print("  1-D PDPs complete.")


# =============================================================================
# FIGURE 2 — 2-D INTERACTION (HEATMAP) PLOTS
# One figure per variable pair; subplots = one per model (5 subplots).
# =============================================================================
def plot_pdp_2d(fitted: dict, X_sc: np.ndarray, X_df: pd.DataFrame,
                scaler: StandardScaler):
    print("\n[Figure 2] 2-D Interaction (PDP surface) plots ...")

    cmap   = "RdYlGn_r"   # green = low risk, red = high risk
    levels = np.linspace(0, 1, 21)

    for (feat1, feat2) in TOP_PAIRS:
        fig, axes = plt.subplots(1, 5, figsize=(26, 5))
        fig.suptitle(
            f"2-D Partial Dependence (Interaction) — {feat1} × {feat2}\n"
            f"(joint marginal effect on P(Risky), averaged over all other features)",
            fontsize=13, fontweight="bold", y=1.04,
        )

        for ax, mname in zip(axes, ALL_MODELS):
            if mname == "ANN":
                g1_sc, g2_sc, surf = pdp_2d_ann(
                    fitted[mname], X_sc, feat1, feat2)
            else:
                g1_sc, g2_sc, surf = pdp_2d_sklearn(
                    fitted[mname], X_df, feat1, feat2)

            # Back-transform axes
            g1_orig = unscale_grid(g1_sc, feat1, scaler)
            g2_orig = unscale_grid(g2_sc, feat2, scaler)

            # surf shape: (len(g1), len(g2))
            # pcolormesh needs (cols=g2) on x-axis, (rows=g1) on y-axis
            im = ax.contourf(
                g2_orig, g1_orig, surf,
                levels=levels, cmap=cmap, vmin=0, vmax=1,
            )
            ax.contour(
                g2_orig, g1_orig, surf,
                levels=[0.5], colors="black",
                linewidths=1.5, linestyles="--",
            )
            ax.set_title(mname, fontsize=10, fontweight="bold")
            ax.set_xlabel(feat2, fontsize=9)
            ax.set_ylabel(feat1 if ax == axes[0] else "", fontsize=9)
            ax.spines[["top", "right"]].set_visible(False)

        # Shared colorbar
        cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.70])
        sm      = plt.cm.ScalarMappable(cmap=cmap,
                                        norm=plt.Normalize(vmin=0, vmax=1))
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cbar_ax)
        cb.set_label("P(Risky)", fontsize=10)
        cb.ax.axhline(0.5, color="black", linewidth=1.5, linestyle="--")
        cb.ax.text(1.6, 0.5, "0.5 threshold", va="center",
                   fontsize=7.5, transform=cb.ax.transData)

        pair_label = f"{feat1.replace('.','')}_x_{feat2.replace('.','')}"
        savefig(fig, "pdp_2d", f"PDP2D_{pair_label}")
        plt.close(fig)
        gc.collect()

    # Combined: rows = pairs, cols = models
    n_pairs = len(TOP_PAIRS)
    fig, axes = plt.subplots(n_pairs, len(ALL_MODELS),
                             figsize=(26, 5.5 * n_pairs))
    fig.suptitle(
        "2-D Partial Dependence Plots — All Pairs × All Models\n"
        "(joint marginal effect on P(Risky), dashed contour = 0.5 threshold)",
        fontsize=14, fontweight="bold", y=1.02,
    )

    for row, (feat1, feat2) in enumerate(TOP_PAIRS):
        for col, mname in enumerate(ALL_MODELS):
            ax = axes[row][col]
            if mname == "ANN":
                g1_sc, g2_sc, surf = pdp_2d_ann(
                    fitted[mname], X_sc, feat1, feat2)
            else:
                g1_sc, g2_sc, surf = pdp_2d_sklearn(
                    fitted[mname], X_df, feat1, feat2)
            g1_orig = unscale_grid(g1_sc, feat1, scaler)
            g2_orig = unscale_grid(g2_sc, feat2, scaler)
            ax.contourf(g2_orig, g1_orig, surf,
                        levels=levels, cmap=cmap, vmin=0, vmax=1)
            ax.contour(g2_orig, g1_orig, surf,
                       levels=[0.5], colors="black",
                       linewidths=1.2, linestyles="--")
            if row == 0:
                ax.set_title(mname, fontsize=10, fontweight="bold")
            if col == 0:
                ax.set_ylabel(f"{feat1}\n↕", fontsize=9)
            ax.set_xlabel(feat2 if row == n_pairs - 1 else "", fontsize=8)
            ax.spines[["top", "right"]].set_visible(False)

    # Global colorbar
    fig.subplots_adjust(right=0.90)
    cbar_ax = fig.add_axes([0.92, 0.10, 0.015, 0.80])
    sm      = plt.cm.ScalarMappable(cmap=cmap,
                                    norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label("P(Risky)", fontsize=11)
    cb.ax.axhline(0.5, color="black", linewidth=1.5, linestyle="--")

    savefig(fig, "pdp_2d", "PDP2D_combined")
    plt.close(fig)
    gc.collect()
    print("  2-D interaction plots complete.")


# =============================================================================
# MAIN
# =============================================================================
def main():
    print("TensorFlow:", tf.__version__)
    set_seed(RANDOM_STATE)
    setup_dirs()

    # Load & scale
    X, y = load_data()
    scaler  = StandardScaler()
    X_sc    = scaler.fit_transform(X)
    X_df    = pd.DataFrame(X_sc, columns=FEATURES)

    # Fit all models on full data
    fitted = fit_all_models(X_sc, y)

    # 1-D PDPs
    plot_pdp_1d(fitted, X_sc, X_df, scaler)

    # 2-D interaction surfaces
    plot_pdp_2d(fitted, X_sc, X_df, scaler)

    print(f"\nAll done. Figures saved to: {os.path.abspath(OUTPUT_DIR)}/figures/")


if __name__ == "__main__":
    main()
